# Neo-Substrate Discovery for Molecular Glue Ternary Complexes

This notebook demonstrates how to identify Neo-substrates that can form productive ternary complexes with molecular glues and E3 ligases.

## Overview

The workflow follows a computational proteomics approach:

1. **Interface Analysis** - Characterize the molecular glue binding interface
2. **Pharmacophore Generation** - Extract key interaction features
3. **Proteome Scanning** - Screen for compatible protein surfaces
4. **Ternary Docking** - Validate candidate binding
5. **Scoring & Ranking** - Prioritize candidates for validation

## Requirements

```bash
conda activate autodock  # or openmm environment
```

In [ ]:
# Standard imports
import os
import sys
from pathlib import Path
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path for imports
module_path = Path.cwd().parent
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print(f"Working directory: {Path.cwd()}")

In [ ]:
# Import Neo-substrate modules
from neosubstrate import (
    NeoSubstratePipeline,
    InterfaceAnalyzer,
    PharmacophoreGenerator,
    SurfaceScanner,
    TernaryDocker,
    NeoSubstrateScorer,
)

from neosubstrate.pharmacophore import (
    pharmacophore_to_vector,
    calculate_pharmacophore_similarity,
)

from neosubstrate.surface_scanner import scan_proteome_for_neo_substrates

print("Neo-substrate modules loaded successfully!")

## 1. Setup: Define Input Structures

We need:
- E3 ligase structure (e.g., CRBN, VHL)
- Molecular glue structure (e.g., thalidomide, lenalidomide)
- Optional: Known ternary complex for reference

In [ ]:
# Define paths to your structures
# Update these paths to point to your actual files

E3_STRUCTURE = "examples/crbn.pdb"  # E3 ligase (e.g., Cereblon)
GLUE_STRUCTURE = "examples/thalidomide.sdf"  # Molecular glue
TERNARY_STRUCTURE = None  # Optional: known ternary complex

# Proteome to scan (directory of PDB files)
PROTEOME_PATH = "examples/proteome/"

# Output directory
OUTPUT_DIR = "results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration set!")

## 2. Interface Analysis

Analyze the molecular glue binding interface to understand:
- Which residues contact the glue
- Interaction types (hydrophobic, H-bond, etc.)
- Interface area and geometry

In [ ]:
# Example: Analyze a ternary complex interface
# (Skip this cell if you don't have a ternary structure)

if TERNARY_STRUCTURE and os.path.exists(TERNARY_STRUCTURE):
    analyzer = InterfaceAnalyzer(
        structure_path=TERNARY_STRUCTURE,
        glue_selection="resname LIG",  # Adjust based on your glue residue name
        e3_selection="chain A",  # E3 ligase chain
        substrate_selection="chain B",  # Substrate chain
        distance_cutoff=5.0
    )
    
    interface = analyzer.analyze()
    
    print(f"E3 contact residues: {len(interface.e3_contacts)}")
    print(f"Substrate contact residues: {len(interface.substrate_contacts)}")
    print(f"Buried surface area: {interface.buried_surface_area:.1f} Å²")
    print(f"Hydrogen bonds: {len(interface.hydrogen_bonds)}")
    print(f"Hydrophobic contacts: {interface.hydrophobic_contacts}")
    
    # Show contact residues
    print("\nE3 contact residues:")
    for contact in interface.e3_contacts[:10]:
        print(f"  {contact.resname}{contact.resid} ({contact.chain}) - {contact.min_distance:.2f} Å")
else:
    print("No ternary structure provided - will use E3 structure for analysis")

## 3. Pharmacophore Generation

Generate a pharmacophore model representing the key interaction features required for Neo-substrate binding.

In [ ]:
# Generate pharmacophore from interface or glue molecule
pharma_gen = PharmacophoreGenerator(feature_radius=1.5)

# Option 1: From interface contacts (if available)
if 'interface' in dir() and interface is not None:
    contacts = [
        {
            'resid': c.resid,
            'resname': c.resname,
            'min_distance': c.min_distance
        }
        for c in interface.substrate_contacts
    ]
    reference_pharmacophore = pharma_gen.from_interface_contacts(contacts)
    print(f"Generated pharmacophore from {len(contacts)} interface contacts")

# Option 2: From molecular glue structure
elif os.path.exists(GLUE_STRUCTURE):
    reference_pharmacophore = pharma_gen.from_molecule(GLUE_STRUCTURE)
    print(f"Generated pharmacophore from glue molecule")

else:
    # Create a sample pharmacophore for demonstration
    from neosubstrate.pharmacophore import Pharmacophore, PharmacophoreFeature, FeatureType
    
    reference_pharmacophore = Pharmacophore(name="demo_pharmacophore")
    # Add sample features
    reference_pharmacophore.add_feature(PharmacophoreFeature(
        feature_type=FeatureType.HYDROPHOBIC,
        position=(0.0, 0.0, 0.0),
        weight=1.0
    ))
    reference_pharmacophore.add_feature(PharmacophoreFeature(
        feature_type=FeatureType.HBOND_ACCEPTOR,
        position=(3.0, 0.0, 0.0),
        weight=1.0
    ))
    print("Created demo pharmacophore")

# Show pharmacophore features
print(f"\nPharmacophore features: {len(reference_pharmacophore.features)}")
feature_counts = {}
for feat in reference_pharmacophore.features:
    ft = feat.feature_type.value
    feature_counts[ft] = feature_counts.get(ft, 0) + 1

for ft, count in sorted(feature_counts.items()):
    print(f"  {ft}: {count}")

In [ ]:
# Convert pharmacophore to feature vector for comparison
ref_vector = pharmacophore_to_vector(reference_pharmacophore)

print(f"Pharmacophore vector shape: {ref_vector.shape}")
print(f"Feature counts: {ref_vector[:7]}")
print(f"Weighted counts: {ref_vector[7:14]}")
print(f"Centroid: {ref_vector[14:17]}")

## 4. Surface Scanning

Scan protein structures to find surface patches that match the pharmacophore requirements.

In [ ]:
# Initialize the surface scanner
scanner = SurfaceScanner(
    sasa_threshold=10.0,  # Minimum SASA for surface residues
    patch_distance=8.0,   # Max distance for clustering
    min_patch_size=3,     # Minimum residues per patch
    max_patch_size=20     # Maximum residues per patch
)

print("Surface scanner initialized")

In [ ]:
# Example: Analyze a single protein
# Replace with path to your protein structure

EXAMPLE_PROTEIN = "examples/target_protein.pdb"

if os.path.exists(EXAMPLE_PROTEIN):
    surface = scanner.analyze_protein(EXAMPLE_PROTEIN)
    
    print(f"Protein: {surface.protein_id}")
    print(f"Total SASA: {surface.total_sasa:.1f} Å²")
    print(f"Surface residues: {len(surface.surface_residues)}")
    print(f"Surface patches: {len(surface.patches)}")
    
    # Find compatible patches
    compatible = scanner.find_compatible_patches(
        surface,
        reference_pharmacophore,
        similarity_threshold=0.3
    )
    
    print(f"\nCompatible patches: {len(compatible)}")
    for patch, score in compatible[:5]:
        print(f"  Patch {patch.patch_id}: similarity={score:.3f}, area={patch.area:.1f} Å²")
else:
    print(f"Example protein not found at {EXAMPLE_PROTEIN}")
    print("Skipping single protein analysis...")

In [ ]:
# Scan multiple proteins (proteome scanning)
if os.path.exists(PROTEOME_PATH):
    proteome_files = list(Path(PROTEOME_PATH).glob("*.pdb"))
    print(f"Found {len(proteome_files)} protein structures")
    
    if proteome_files:
        # Scan proteome
        hits = scan_proteome_for_neo_substrates(
            glue_pharmacophore=reference_pharmacophore,
            proteome_structures=[str(f) for f in proteome_files],
            similarity_threshold=0.4,
            n_workers=2
        )
        
        print(f"\nFound {len(hits)} Neo-substrate hits")
        
        # Show top hits
        print("\nTop 10 hits:")
        for hit in hits[:10]:
            print(f"  {hit.protein_id}: similarity={hit.similarity_score:.3f}")
else:
    print(f"Proteome path not found: {PROTEOME_PATH}")
    hits = []

## 5. Ternary Complex Docking

Dock top candidates to validate ternary complex formation.

In [ ]:
# Initialize the ternary docker
if os.path.exists(E3_STRUCTURE) and os.path.exists(GLUE_STRUCTURE):
    docker = TernaryDocker(
        e3_pdb=E3_STRUCTURE,
        glue_sdf=GLUE_STRUCTURE,
        box_size=(30.0, 30.0, 30.0),
        exhaustiveness=16,
        n_poses=10
    )
    print("Ternary docker initialized")
else:
    docker = None
    print("E3 or glue structure not found - skipping docking setup")

In [ ]:
# Dock top candidates
docking_results = {}

if docker and hits:
    # Dock top 5 candidates (adjust as needed)
    for hit in hits[:5]:
        print(f"Docking {hit.protein_id}...")
        try:
            result = docker.dock_substrate(hit.structure_path)
            docking_results[hit.protein_id] = result
            
            best = result.get_best_pose()
            if best:
                print(f"  Best score: {best.score:.2f} kcal/mol")
        except Exception as e:
            print(f"  Docking failed: {e}")
else:
    print("Skipping docking (no docker or no hits)")

## 6. Scoring and Ranking

Combine all scores to rank Neo-substrate candidates.

In [ ]:
# Initialize scorer
from neosubstrate.scoring import ScoringWeights

weights = ScoringWeights(
    pharmacophore=0.30,  # Pharmacophore similarity
    docking=0.25,        # Docking score
    interface=0.20,      # Interface quality
    druggability=0.15,   # Binding site druggability
    structural=0.10,     # Structural features
)

scorer = NeoSubstrateScorer(
    reference_pharmacophore=reference_pharmacophore,
    weights=weights
)

print("Scorer initialized with custom weights")

In [ ]:
# Score and rank candidates
if hits:
    scores = scorer.score_candidates(hits, docking_results)
    ranked = scorer.rank(scores)
    
    print(f"Scored {len(ranked)} candidates\n")
    print("Top 10 Neo-substrate candidates:")
    print("-" * 70)
    print(f"{'Rank':<6}{'Protein ID':<20}{'Total':<10}{'Pharm':<10}{'Dock':<10}")
    print("-" * 70)
    
    for score in ranked[:10]:
        print(
            f"{score.rank:<6}"
            f"{score.candidate_id:<20}"
            f"{score.total_score:<10.3f}"
            f"{score.pharmacophore_score:<10.3f}"
            f"{score.docking_score:<10.3f}"
        )
else:
    print("No hits to score")
    ranked = []

## 7. Using the Complete Pipeline

The `NeoSubstratePipeline` class provides a convenient way to run the entire workflow.

In [ ]:
# Run the complete pipeline
if os.path.exists(E3_STRUCTURE) and os.path.exists(GLUE_STRUCTURE):
    pipeline = NeoSubstratePipeline(
        e3_structure=E3_STRUCTURE,
        glue_structure=GLUE_STRUCTURE,
        ternary_structure=TERNARY_STRUCTURE
    )
    
    # Run if proteome exists
    if os.path.exists(PROTEOME_PATH):
        candidates = pipeline.run(
            proteome_path=PROTEOME_PATH,
            top_k=50,
            dock=True,
            validate=True
        )
        
        # Export results
        pipeline.export_results(
            os.path.join(OUTPUT_DIR, "neo_substrates.csv"),
            format="csv"
        )
        
        # Print summary
        summary = pipeline.get_summary()
        print("\nPipeline Summary:")
        for key, value in summary.items():
            print(f"  {key}: {value}")
    else:
        print(f"Proteome path not found: {PROTEOME_PATH}")
else:
    print("Required structures not found - cannot run pipeline")

## 8. Visualizing Results

Visualize the identified Neo-substrate binding patches.

In [ ]:
# Try to import visualization libraries
try:
    import py3Dmol
    HAS_PY3DMOL = True
except ImportError:
    HAS_PY3DMOL = False
    print("py3Dmol not available - skipping 3D visualization")

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not available - skipping plots")

In [ ]:
# Plot score distribution
if HAS_MATPLOTLIB and ranked:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Total scores
    total_scores = [s.total_score for s in ranked]
    axes[0].hist(total_scores, bins=20, edgecolor='black')
    axes[0].set_xlabel('Total Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Total Score Distribution')
    
    # Pharmacophore scores
    pharm_scores = [s.pharmacophore_score for s in ranked]
    axes[1].hist(pharm_scores, bins=20, edgecolor='black', color='orange')
    axes[1].set_xlabel('Pharmacophore Score')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Pharmacophore Similarity')
    
    # Score components for top candidates
    top_n = min(10, len(ranked))
    labels = [s.candidate_id[:10] for s in ranked[:top_n]]
    x = np.arange(top_n)
    width = 0.35
    
    axes[2].bar(x - width/2, [s.pharmacophore_score for s in ranked[:top_n]], width, label='Pharmacophore')
    axes[2].bar(x + width/2, [s.docking_score for s in ranked[:top_n]], width, label='Docking')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(labels, rotation=45, ha='right')
    axes[2].set_ylabel('Score')
    axes[2].set_title('Top Candidates Score Breakdown')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'score_distribution.png'), dpi=150)
    plt.show()

In [ ]:
# 3D visualization of a hit (if py3Dmol available)
if HAS_PY3DMOL and hits:
    top_hit = hits[0]
    
    if os.path.exists(top_hit.structure_path):
        with open(top_hit.structure_path, 'r') as f:
            pdb_data = f.read()
        
        view = py3Dmol.view(width=800, height=500)
        view.addModel(pdb_data, 'pdb')
        
        # Style protein
        view.setStyle({'cartoon': {'color': 'lightgray'}})
        
        # Highlight binding patch residues
        patch_resids = [r.resid for r in top_hit.patch.residues]
        view.addStyle(
            {'resi': patch_resids},
            {'stick': {'color': 'red'}, 'cartoon': {'color': 'red'}}
        )
        
        view.zoomTo()
        view.show()
        
        print(f"Visualizing {top_hit.protein_id}")
        print(f"Red residues = predicted binding patch")
    else:
        print(f"Structure file not found: {top_hit.structure_path}")

## 9. Export and Next Steps

Export results and prepare for experimental validation.

In [ ]:
# Export detailed results
import json

if ranked:
    results_data = {
        'pipeline_info': {
            'e3_structure': E3_STRUCTURE,
            'glue_structure': GLUE_STRUCTURE,
            'n_candidates': len(ranked)
        },
        'candidates': [s.to_dict() for s in ranked[:50]]
    }
    
    output_json = os.path.join(OUTPUT_DIR, 'neo_substrate_results.json')
    with open(output_json, 'w') as f:
        json.dump(results_data, f, indent=2, default=str)
    
    print(f"Results exported to {output_json}")

In [ ]:
# Summary and recommendations
print("="*60)
print("NEO-SUBSTRATE DISCOVERY SUMMARY")
print("="*60)

if ranked:
    print(f"\nTotal candidates identified: {len(ranked)}")
    print(f"Top scoring candidate: {ranked[0].candidate_id}")
    print(f"  - Total score: {ranked[0].total_score:.3f}")
    print(f"  - Pharmacophore match: {ranked[0].pharmacophore_score:.3f}")
    
    # Count high-confidence candidates
    high_conf = sum(1 for s in ranked if s.total_score > 0.7)
    med_conf = sum(1 for s in ranked if 0.5 <= s.total_score <= 0.7)
    
    print(f"\nConfidence breakdown:")
    print(f"  - High confidence (>0.7): {high_conf}")
    print(f"  - Medium confidence (0.5-0.7): {med_conf}")
    
    print("\nNext steps:")
    print("  1. Validate top candidates with MD simulations")
    print("  2. Run FEP calculations for binding affinity")
    print("  3. Experimental validation (pulldown, degradation assay)")
else:
    print("\nNo candidates found. Consider:")
    print("  1. Lowering similarity threshold")
    print("  2. Expanding the proteome search set")
    print("  3. Refining the reference pharmacophore")

---

## Additional Resources

- **Interface Analysis**: See `interface_analysis.py` for detailed contact analysis
- **Pharmacophore Models**: See `pharmacophore.py` for custom feature definitions
- **Surface Scanning**: See `surface_scanner.py` for proteome-scale analysis
- **Docking**: See `ternary_dock.py` for constrained docking options
- **Scoring**: See `scoring.py` for ML-based scoring models